# YOLO Model

## Import Libraries

In [1]:
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
import sys

from ultralytics import YOLO

## Define necessary func

In [2]:
def letterbox_image(image, size=(640, 640)):
    h, w, _ = image.shape
    desired_size = max(size)

    # scale the image so its maximum dimension matches our desired size
    scale = desired_size / max(h, w)
    resized_image = cv2.resize(image, (round(w*scale), round(h*scale)), interpolation=cv2.INTER_LINEAR)

    # create a black canvas of desired size
    letterbox = np.full((desired_size, desired_size, 3), 0)

    # compute center offset
    dh, dw, _ = resized_image.shape
    start = [(desired_size-dh)//2, (desired_size-dw)//2]

    # copy the resized image into center of letterbox
    letterbox[start[0]:start[0]+dh, start[1]:start[1]+dw, :] = resized_image

    return letterbox.astype(np.uint8)

In [3]:
def draw_rec(img, x_min, y_min, x_max, y_max):
    draw_img = img
    # Blue color in BGR
    color = (255, 0, 0)
    # Line thickness
    thickness = 3
    
    # draw_img = cv2.line(draw_img, (x_min, y_min), (x_max, y_min), color, thickness)
    # draw_img = cv2.line(draw_img, (x_min, y_min), (x_min, y_max), color, thickness)
    # draw_img = cv2.line(draw_img, (x_max, y_min), (x_max, y_max), color, thickness)
    # draw_img = cv2.line(draw_img, (x_min, y_max), (x_max, y_max), color, thickness)
    
    cv2.rectangle(draw_img, (x_min, y_min), (x_max, y_max), color, thickness)
    
    return draw_img

## Load model

In [ ]:
YOLOmodel_detect = YOLO('../model/yolo11x_kidneyDetect_old.pt')
YOLOmodel_segment = YOLO('../model/yolo11x_stonesSegment.pt')
# YOLOmodel_segment = YOLO('../model/09-Nov/yolo11x_stonesSegment_noNULL.pt')

# Do the test

In [10]:
src_path = r'D:\University\LuanVan\Kidney_stone_detection-main\Dataset\Train\Kidney_stone'
# src_path = r'C:\Users\ASUS TUF\Pictures\Screenshots'
btn = 0

for file_name in os.listdir(src_path):
    cv2.destroyAllWindows()
    if btn == 27:
        break
    btn = 0
    
    if not (file_name.endswith('.jpg') or file_name.endswith('.png')):
        continue
    
    img_path = os.path.join(src_path, file_name)
    img_path = os.path.normpath(img_path)

    img = cv2.imread(img_path)
    show_img = letterbox_image(img, (400, 400))
    cv2.imshow('CT Image', show_img)
    
    results = YOLOmodel_detect(img, max_det=2, conf=0.8, device=0)
    
    if len(results[0].boxes.xyxy) <= 0:
        continue
    
    draw_img = img.copy()
    for coordinate in results[0].boxes.xyxy:
        x_min, y_min, x_max, y_max = coordinate.tolist()
        draw_img = draw_rec(draw_img, int(x_min), int(y_min), int(x_max), int(y_max))
        show_draw_img = letterbox_image(draw_img, (400, 400))
        cv2.imshow('Kidney', show_draw_img)
        
        img_cropped = img[int(y_min):int(y_max), int(x_min):int(x_max)]
        # img_cropped = letterbox_image(img_cropped)
        cv2.imshow('Image', img_cropped)
        
        segmentResults = YOLOmodel_segment(img_cropped, conf=0.2, device=0)
        # masks = segmentResults[0].masks
        # boxes = segmentResults[0].boxes
        
        # if masks is not None:
        #     xy_coords = masks.xy
            
        #     # Create binary mask
        #     b_mask = np.zeros(img.shape[:2], np.uint8)
            
        #     # Extract contour result
        #     contour = xy_coords.pop()

        #     # Changing the type
        #     contour = contour.astype(np.int32)

        #     # Reshaping
        #     contour = contour.reshape(-1, 1, 2)
        #     # Draw contour onto mask
        #     _ = cv2.drawContours(b_mask, [contour], -1, (255, 255, 255), cv2.FILLED)

        #     cv2.imshow('Contour', b_mask)
        #     pixel_count = np.sum(b_mask >= 200)
            
        #     print(pixel_count)
        #     print(b_mask.shape)
        #     print(type(xy_coords))
        # if hasattr(masks, 'xy'): # check if masks object has attribue 'xy' (also check if masks is not None)
        # if masks is not None and hasattr(masks, 'xy') and boxes is not None and hasattr(boxes, 'xyxy'):
            # xy_coords = masks.xy
            # xyxy_coords = boxes.xyxy
            
            # for i, (coords, box_coords) in enumerate(zip(xy_coords, xyxy_coords), start=1):
            #     coords_int = np.round(coords).astype(int)
            #     unique_coordsPairs = set(map(tuple, coords_int))
                
            #     # Draw points on the image
            #     for pair in unique_coordsPairs:
            #         x, y = pair
            #         # Make sure the points stay within the bounds of the image
            #         if 0 <= x < img_cropped.shape[1] and 0 <= y < img_cropped.shape[0]:
            #             cv2.circle(img_cropped, (x, y), 0, (255, 0, 0), -1)
                
            #     num_coords = len(unique_coordsPairs)
            #     print(f"Mask {i + 1} has {num_coords} coordinates and in box {box_coords}.")
                
        btn = cv2.waitKey(0)
        if btn == 27: # Esc
            break
            


0: 640x512 2 Kidneys, 277.0ms
Speed: 12.0ms preprocess, 277.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x448 (no detections), 72.0ms
Speed: 1.0ms preprocess, 72.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 448)

0: 640x512 1 Stones, 64.0ms
Speed: 2.0ms preprocess, 64.0ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 2 Kidneys, 50.0ms
Speed: 3.0ms preprocess, 50.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x448 (no detections), 60.0ms
Speed: 2.0ms preprocess, 60.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 448)

0: 640x480 2 Stoness, 62.1ms
Speed: 0.9ms preprocess, 62.1ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 480)

0: 640x512 2 Kidneys, 50.0ms
Speed: 3.0ms preprocess, 50.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x480 1 Stones, 63.0ms
Speed: 1.0ms preprocess, 63.0ms inference, 3.0ms postprocess per ima

In [6]:
src_path = r'D:\University\LuanVan\Kidney_stone_detection-main\Dataset\Train\Kidney_stone'
# src_path = r'C:\Users\ASUS TUF\Pictures\Screenshots'
btn = 0

def count_pixels_from_mask(mask):
    # Convert mask to integer coordinates (optional, depending on OpenCV version)
    mask_int = np.round(mask).astype(np.int32)

    # Create a blank image
    min_x, min_y = mask_int.min(axis=0)
    max_x, max_y = mask_int.max(axis=0)
    img_size = (max_x - min_x + 1, max_y - min_y + 1)
    img = np.zeros(img_size, dtype=np.uint8)

    # Draw the mask on the image
    cv2.fillPoly(img, [mask_int - min_x], color=255)

    # Count non-zero pixels
    pixel_count = cv2.countNonZero(img)
    return pixel_count

def shoelace_formula(points):
    x = [p[0] for p in points]
    y = [p[1] for p in points]

    n = len(x)
    area = 0.5 * sum(x[i] * y[(i + 1) % n] - x[(i + 1) % n] * y[i] for i in range(n))
    return abs(area)

for file_name in os.listdir(src_path):
    cv2.destroyAllWindows()
    if btn == 27:
        break
    btn = 0
    
    if not (file_name.endswith('.jpg') or file_name.endswith('.png')):
        continue
    
    img_path = os.path.join(src_path, file_name)
    img_path = os.path.normpath(img_path)

    img = cv2.imread(img_path)
    show_img = letterbox_image(img, (400, 400))
    cv2.imshow('CT Image', show_img)
    
    results = YOLOmodel_detect(img, max_det=2, conf=0.8, device=0)
    
    if len(results[0].boxes.xyxy) <= 0:
        continue
    
    draw_img = img.copy()
    for coordinate in results[0].boxes.xyxy:
        x_min, y_min, x_max, y_max = coordinate.tolist()
        draw_img = draw_rec(draw_img, int(x_min), int(y_min), int(x_max), int(y_max))
        show_draw_img = letterbox_image(draw_img, (400, 400))
        cv2.imshow('Kidney', show_draw_img)
        
        img_cropped = img[int(y_min):int(y_max), int(x_min):int(x_max)]
        # img_cropped = letterbox_image(img_cropped)
        cv2.imshow('Image', img_cropped)
        
        segmentResults = YOLOmodel_segment(img_cropped, conf=0.5, device=0, show=True)
        masks = segmentResults[0].masks
        boxes = segmentResults[0].boxes
        
        if hasattr(masks, 'xy') and hasattr(boxes, 'xyxy'):
            xy_masks = masks.xy
            xyxy_boxes = boxes.xyxy
        
            print(f'Masks xy: {xy_masks}')
            print(f'Masks xy type: {type(xy_masks)}')
            print(f'Boxes xyxy: {xyxy_boxes}')
            
            area = shoelace_formula(xy_masks[0])
            print(f"Area of the polygon: {area:.2f}")
            
            # Nếu xy_masks là danh sách, chuyển đổi thành NumPy array
            if isinstance(xy_masks, list):
                xy_masks = np.vstack(xy_masks)  # Chuyển danh sách thành mảng NumPy
                
            print(f'Masks xy NumPy array: {xy_masks}')
            
            coords_int = np.round(xy_masks).astype(int)
            unique_coordsPairs = set(map(tuple, coords_int))
                            
        btn = cv2.waitKey(0)
        if btn == 27: # Esc
            break
            


0: 640x512 2 Kidneys, 365.3ms
Speed: 9.0ms preprocess, 365.3ms inference, 3.3ms postprocess per image at shape (1, 3, 640, 512)

0: 640x448 (no detections), 71.2ms
Speed: 2.0ms preprocess, 71.2ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 448)

0: 640x512 1 Stones, 63.0ms
Speed: 1.6ms preprocess, 63.0ms inference, 10.5ms postprocess per image at shape (1, 3, 640, 512)
Masks xy: [array([[     109.57,      143.52],
       [     109.57,      144.66],
       [     108.81,      145.42],
       [     108.43,      145.42],
       [     107.67,      146.18],
       [     107.29,      146.18],
       [     105.39,      148.08],
       [     103.87,      148.08],
       [     103.87,      160.61],
       [     105.01,      160.61],
       [     105.77,      161.37],
       [     106.15,      161.37],
       [     107.29,      162.51],
       [     107.67,      162.51],
       [     108.43,      163.27],
       [     108.81,      163.27],
       [     109.19,      163.65],
     

In [ ]:
from ultralytics import YOLO
# Load a model
model = YOLO("yolo11n-seg.pt")
# Validate the model
metrics = model.val()
print("Mean Average Precision for boxes:", metrics.box.map)
print("Mean Average Precision for masks:", metrics.seg.map)

In [5]:
import numpy as np

# Dữ liệu xy từ masks
masks_xy = np.array([
    [109.57, 143.52],
    [109.57, 144.66],
    [108.81, 145.42],
    [108.43, 145.42],
    [107.67, 146.18],
    [107.29, 146.18],
    [105.39, 148.08],
    [103.87, 148.08],
    [103.87, 160.61],
    [105.01, 160.61],
    [105.77, 161.37],
    [106.15, 161.37],
    [107.29, 162.51],
    [107.67, 162.51],
    [108.43, 163.27],
    [108.81, 163.27],
    [109.19, 163.65],
    [109.57, 163.65],
    [110.71, 164.78],
    [111.09, 164.78],
    [111.85, 165.54],
    [111.85, 165.92],
    [112.22, 165.92],
    [112.98, 166.68],
    [113.36, 166.68],
    [113.74, 167.06],
    [114.12, 167.06],
    [114.5, 167.44],
    [117.54, 167.44],
    [117.92, 167.06],
    [119.06, 167.06],
    [119.44, 166.68],
    [119.82, 166.68],
    [120.2, 166.3],
    [120.58, 166.3],
    [121.34, 165.54],
    [121.72, 165.54],
    [122.86, 164.4],
    [122.86, 164.02],
    [125.51, 161.37],
    [125.51, 160.99],
    [125.89, 160.61],
    [125.89, 160.23],
    [126.65, 159.47],
    [127.79, 159.47],
    [127.79, 152.63],
    [126.65, 152.63],
    [126.27, 152.25],
    [126.27, 151.5],
    [125.89, 151.12],
    [125.89, 149.98],
    [125.51, 149.6],
    [125.51, 149.22],
    [124.75, 148.46],
    [124.75, 148.08],
    [123.24, 146.56],
    [123.24, 146.18],
    [121.72, 144.66],
    [121.72, 143.52]
], dtype=np.float32)

# Công thức Shoelace
def calculate_polygon_area(polygon):
    x = polygon[:, 0]
    y = polygon[:, 1]
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

# Tính diện tích
area = calculate_polygon_area(masks_xy)
print(f"Diện tích vật thể: {area:.2f} pixel^2")

Diện tích vật thể: 445.81 pixel^2


In [ ]:
import cv2
import numpy as np
# Create binary mask
b_mask = np.zeros(img.shape[:2], np.uint8)

# Extract contour result
contour = c.masks.xy.pop()

# Changing the type
contour = contour.astype(np.int32)

# Reshaping
contour = contour.reshape(-1, 1, 2)
# Draw contour onto mask
_ = cv2.drawContours(b_mask, [contour], -1, (255, 255, 255), cv2.FILLED)

In [14]:
import pydicom
import cv2
import numpy as np

from aspose.imaging import Image
from aspose.imaging.imageoptions import JpegOptions

from ultralytics import YOLO
model = YOLO('../model/07-Nov/yolo11x_kidneyDetect.pt')

# Đường dẫn đến file DICOM
dicom_path = r"C:\Users\ASUS TUF\Documents\Kidney Stone Dataset\manifest-1592488683281\C4KC-KiTS\KiTS-00036\07-23-2004-NA-twophaseabdomenpelvis-41151\2.000000-arterial-16785\1-047.dcm"

# Đọc file DICOM
dicom_data = pydicom.dcmread(dicom_path)

if 'PixelSpacing' in dicom_data:
    pixel_spacing = dicom_data.PixelSpacing
    print(f"Pixel Spacing: {pixel_spacing}")

jpeg_options = JpegOptions()

image = Image.load(dicom_path)
image.save('temp.jpg', jpeg_options)

img = cv2.imread('temp.jpg')

results = model(img, max_det=2, show=True)
cv2.waitKey(0)
cv2.destroyAllWindows()

Pixel Spacing: [0.798828125, 0.798828125]

0: 640x640 2 Kidneys, 687.0ms
Speed: 3.0ms preprocess, 687.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)


In [16]:
import numpy as np
import cv2

# Your list of polygon coordinates (float type)
polygon = [
    [105.47, 160.28],
    [105.47, 161.36],
    [104.75, 162.08],
    [104.39, 162.08],
    [104.03, 162.44],
    [102.59, 162.44],
    [102.59, 169.98],
    [103.67, 169.98],
    [104.75, 171.06],
    [104.75, 171.42],
    [106.55, 173.22],
    [106.55, 174.3],
    [111.58, 174.3],
    [111.58, 173.22],
    [111.94, 172.86],
    [112.3, 172.86],
    [112.66, 172.5],
    [113.02, 172.5],
    [113.73, 171.78],
    [114.09, 171.78],
    [114.81, 171.06],
    [114.81, 170.7],
    [115.53, 169.98],
    [116.61, 169.98],
    [116.61, 167.47],
    [115.53, 167.47],
    [115.17, 167.11],
    [115.17, 163.88],
    [114.81, 163.52],
    [114.81, 162.08],
    [114.45, 161.72],
    [114.45, 160.28]
]

# Convert to integer pixel coordinates
polygon_int = np.round(polygon).astype(np.int32)

# Create a blank image (large enough to fit the polygon)
img_shape = (200, 200)  # You can adjust the size as needed
blank_image = np.zeros(img_shape, dtype=np.uint8)

# Draw the polygon
cv2.fillPoly(blank_image, [polygon_int], color=255)

# Count non-zero pixels (area of the polygon in pixels)
pixel_count = np.count_nonzero(blank_image)

print(f"Pixel Count: {pixel_count}")

Pixel Count: 177


In [ ]:
import torch, torchvision, numpy
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

print(torchvision.__version__)

print(numpy.__version__)

2.5.1+cu118
True
1
0.20.1+cpu
2.0.2


: 

In [ ]:
test = r'D:\University\LuanVan\dataset_train\Detect Kidney in CT Images.v5i.yolov11\data-test.yaml'
metrics = YOLOmodel_detect.val(data=test, device=0)
print('mAP: ', metrics.box.map)
print('mAP50: ', metrics.box.map50)
print('mAP75: ', metrics.box.map75)
print('mAPs: ', metrics.box.maps)

Ultralytics 8.3.31  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11x summary (fused): 464 layers, 56,828,179 parameters, 0 gradients, 194.4 GFLOPs


val: Scanning D:\University\LuanVan\dataset_train\Detect Kidney in CT Images.v5i.yolov11\test\labels... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<00:00, 505.87it/s]

val: New cache created: D:\University\LuanVan\dataset_train\Detect Kidney in CT Images.v5i.yolov11\test\labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:28<00:00,  4.02s/it]


                   all        100        196      0.995      0.995      0.995       0.88
Speed: 2.4ms preprocess, 199.7ms inference, 0.0ms loss, 5.5ms postprocess per image
Results saved to runs\detect\val
mAP:  0.8795068212260813
mAP50:  0.995
mAP75:  0.9771560020993347
mAPs:  [    0.87951]


In [10]:
test = r'D:\University\LuanVan\LV_TrangThanhPhat_B2007204\dataset\Segment Stones Final\data-test.yaml'
metrics = YOLOmodel_segment.val(data=test, device=0)
print('mAP: ', metrics.box.map)
print('mAP50: ', metrics.box.map50)
print('mAP75: ', metrics.box.map75)
print('mAPs: ', metrics.box.maps)

print('Seg mAP: ', metrics.seg.map)  # map50-95(M)
print('Seg mAP50: ', metrics.seg.map50)  # map50(M)
print('Seg mAP75: ', metrics.seg.map75)  # map75(M)
print('Seg mAPs: ', metrics.seg.maps)  # a list contains map50-95(M) of each category

Ultralytics 8.3.31  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11x-seg summary (fused): 491 layers, 62,003,283 parameters, 0 gradients, 318.5 GFLOPs


val: Scanning D:\University\LuanVan\LV_TrangThanhPhat_B2007204\dataset\Segment Stones Final\test\labels... 100 images, 0 backgrounds, 0 corrupt: 100%|██████████| 100/100 [00:00<00:00, 476.20it/s]

val: New cache created: D:\University\LuanVan\LV_TrangThanhPhat_B2007204\dataset\Segment Stones Final\test\labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:55<00:00,  7.96s/it]


                   all        100        116      0.824      0.871      0.864       0.44      0.837      0.871      0.899      0.475
Speed: 0.6ms preprocess, 539.1ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to runs\segment\val2
mAP:  0.43998701454321687
mAP50:  0.8638775611535012
mAP75:  0.38543883163026005
mAPs:  [    0.43999]
Seg mAP:  0.4751514911131121
Seg mAP50:  0.8992625668282489
Seg mAP75:  0.424715188386749
Seg mAPs:  [    0.47515]


In [11]:
old_YOLOmodel_segment = YOLO('../model/yolo11x_stonesSegment_old.pt')
test = r'D:\University\LuanVan\LV_TrangThanhPhat_B2007204\dataset\segmentStones_notNULL\data-test.yaml'
metrics = old_YOLOmodel_segment.val(data=test, device=0)
print('mAP: ', metrics.box.map)
print('mAP50: ', metrics.box.map50)
print('mAP75: ', metrics.box.map75)
print('mAPs: ', metrics.box.maps)

print('Seg mAP: ', metrics.seg.map)  # map50-95(M)
print('Seg mAP50: ', metrics.seg.map50)  # map50(M)
print('Seg mAP75: ', metrics.seg.map75)  # map75(M)
print('Seg mAPs: ', metrics.seg.maps)  # a list contains map50-95(M) of each category

Ultralytics 8.3.31  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11x-seg summary (fused): 491 layers, 62,003,283 parameters, 0 gradients, 318.5 GFLOPs


val: Scanning D:\University\LuanVan\LV_TrangThanhPhat_B2007204\dataset\segmentStones_notNULL\test\labels... 27 images, 0 backgrounds, 0 corrupt: 100%|██████████| 27/27 [00:00<00:00, 409.08it/s]

val: New cache created: D:\University\LuanVan\LV_TrangThanhPhat_B2007204\dataset\segmentStones_notNULL\test\labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:24<00:00, 12.20s/it]


                   all         27         37      0.841      0.892      0.886      0.545      0.841      0.892      0.882      0.554
Speed: 7.3ms preprocess, 641.8ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to runs\segment\val3
mAP:  0.5445919524596717
mAP50:  0.8859515590017305
mAP75:  0.530679201368521
mAPs:  [    0.54459]
Seg mAP:  0.5535746632198694
Seg mAP50:  0.8822607114541099
Seg mAP75:  0.6203997895879655
Seg mAPs:  [    0.55357]
